<a href="https://colab.research.google.com/github/hiten4/Day33AM/blob/main/Day33_AM_PartA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 33 — Part A: SVM & KNN


## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Utility Functions

In [2]:
def load_data():
    digits = load_digits()
    return digits.data, digits.target

def preprocess(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X), scaler

def split_data(X, y):
    return train_test_split(X, y, test_size=0.2, random_state=42)

def train_svm(X_train, y_train):
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': [0.001, 0.01, 0.1]
    }
    grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=3)
    grid.fit(X_train, y_train)
    return grid.best_estimator_, grid.best_params_

def train_knn(X_train, y_train, X_test, y_test):
    best_k, best_acc = 1, 0

    for k in range(1, 11):
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))

        if acc > best_acc:
            best_k, best_acc = k, acc

    best_model = KNeighborsClassifier(n_neighbors=best_k)
    best_model.fit(X_train, y_train)
    return best_model, best_k

def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)

    print(f"\n{name} Results")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))


## 3. Pipeline Execution

In [3]:
# Load data
X, y = load_data()

# Preprocess
X_scaled, scaler = preprocess(X)

# Split
X_train, X_test, y_train, y_test = split_data(X_scaled, y)

# Train models
svm_model, svm_params = train_svm(X_train, y_train)
print("Best SVM Params:", svm_params)

knn_model, best_k = train_knn(X_train, y_train, X_test, y_test)
print("Best K:", best_k)

# Evaluate
evaluate(svm_model, X_test, y_test, "SVM")
evaluate(knn_model, X_test, y_test, "KNN")


Best SVM Params: {'C': 10, 'gamma': 0.01}
Best K: 1

SVM Results
Accuracy: 0.9805555555555555
Confusion Matrix:
 [[33  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0]
 [ 0  0 33  0  0  0  0  0  0  0]
 [ 0  0  0 33  0  1  0  0  0  0]
 [ 0  0  0  0 46  0  0  0  0  0]
 [ 0  0  0  0  0 45  1  0  0  1]
 [ 0  0  0  0  0  0 35  0  0  0]
 [ 0  0  0  0  0  0  0 33  0  1]
 [ 0  0  1  0  0  0  0  0 29  0]
 [ 0  0  0  1  0  0  0  0  1 38]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        28
           2       0.97      1.00      0.99        33
           3       0.97      0.97      0.97        34
           4       1.00      1.00      1.00        46
           5       0.98      0.96      0.97        47
           6       0.97      1.00      0.99        35
           7       1.00      0.97      0.99        34
           8       0.97      0.97      0.97  

## 4. Confusion Analysis

Analyze confusion matrices to identify commonly confused digits like (3,8), (4,9), (1,7).